# 01 · Qwen3-VL Locally: QA, OCR, Grounding

**Hardware**: 🟡 12GB+ VRAM recommended (default 4B-Instruct is ~9GB in bf16; use the 2B or 4-bit quantization if short on memory)

## What you will learn

1. Run the small sizes of the open-source flagship VLM family: image QA and multi-image comparison
2. OCR / document understanding: what dynamic resolution buys you (cf. [theory.md](../index.md) §2)
3. **Grounding**: make the model output bounding boxes and draw them — the bedrock skill of GUI agents
4. Watch visual token counts track image resolution (cost intuition)

> Version note (2026-08): Qwen3-VL needs a recent transformers (>=4.57). Defer to the [Qwen3-VL GitHub](https://github.com/QwenLM/Qwen3-VL) for current model cards.

In [ ]:
%pip install -q "transformers>=4.57" accelerate torch pillow requests matplotlib
# If VRAM is tight, also: %pip install -q bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"   # short on VRAM? use "Qwen/Qwen3-VL-2B-Instruct"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
    # 4-bit quantization (roughly halves VRAM):
    # quantization_config=__import__('transformers').BitsAndBytesConfig(load_in_4bit=True),
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"loaded {MODEL_ID} on {model.device}")

In [ ]:
# A chat helper we will reuse throughout
def chat(image_or_images, prompt, max_new_tokens=512):
    imgs = image_or_images if isinstance(image_or_images, list) else [image_or_images]
    content = [{"type": "image", "image": im} for im in imgs]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    n_visual = (inputs["input_ids"] == model.config.image_token_id).sum().item()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    reply = processor.batch_decode(
        out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    return reply, n_visual

## 1. Image QA + visual-token cost intuition

Watch the `visual tokens` number in each output: Qwen3-VL patchifies at native resolution (with 2×2 merge compression) — **bigger image, more tokens, more money**.

In [ ]:
import requests
from io import BytesIO
from PIL import Image

def load(url):
    return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")

img_cats = load("http://images.cocodataset.org/val2017/000000039769.jpg")

reply, n_vis = chat(img_cats, "Describe this image. How many cats are there, and what pose is each in?")
print(f"[visual tokens: {n_vis}]\n{reply}")

# Ask again at half size and compare token counts
small = img_cats.resize((img_cats.width // 2, img_cats.height // 2))
_, n_vis_small = chat(small, "How many cats are in this image?")
print(f"\nfull size {img_cats.size}: {n_vis} tokens | half size {small.size}: {n_vis_small} tokens")

Compare with chapter 00's CLIP experiment: CLIP couldn't count cats, but a VLM can — because an LLM is now reasoning on top of the visual features.

## 2. OCR and document understanding

Switch to an image with text (a receipt / screenshot / document photo). Demanding structured output is the production move — **JSON extraction beats free-text transcription**.

In [ ]:
# A Wikimedia receipt photo for demo; your own receipt/screenshot is more fun
img_doc = load("https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Receipt_in_Costa_Rica.jpg/640px-Receipt_in_Costa_Rica.jpg")

reply, n_vis = chat(
    img_doc,
    'This is a receipt. Output JSON: {"merchant": name, "date": date, "total": total, "items": [item list]}. Use null for unreadable fields. Do not guess.',
)
print(f"[visual tokens: {n_vis}]\n{reply}")

**Anti-hallucination habit**: say "use null for unreadable fields, do not guess" explicitly. VLMs confidently invent blurry text — for bulk production use chapter 02's specialized OCR models (grounded, traceable) instead.

## 3. Grounding: output and draw bounding boxes

Grounding = mapping language references to pixel coordinates. Qwen3-VL's detection convention is **0–1000 normalized coordinates**. This ability underpins GUI agents ("click the login button") and robotic grasping.

In [ ]:
import json, re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

reply, _ = chat(
    img_cats,
    'Locate every cat in the image. Output a JSON array: [{"label": "cat", "bbox_2d": [x1, y1, x2, y2]}] with 0-1000 normalized coordinates.',
)
print(reply)

m = re.search(r"\[.*\]", reply, re.S)
boxes = json.loads(m.group()) if m else []

W, H = img_cats.size
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img_cats)
for b in boxes:
    x1, y1, x2, y2 = b["bbox_2d"]
    # 0-1000 normalized -> pixels; skip if the model returned pixel coordinates
    if max(x1, y1, x2, y2) <= 1000 and (x2 <= 1000):
        x1, x2 = x1 / 1000 * W, x2 / 1000 * W
        y1, y2 = y1 / 1000 * H, y2 / 1000 * H
    ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    fill=False, color="red", lw=2))
    ax.text(x1, y1 - 5, b["label"], color="red", fontsize=10)
ax.axis("off")
plt.show()

## 4. Multi-image comparison

Modern VLMs accept interleaved multi-image input — the basis of spot-the-difference, product comparison, and before/after apps.

In [ ]:
img_bear = load("http://images.cocodataset.org/val2017/000000000285.jpg")

reply, n_vis = chat(
    [img_cats, img_bear],
    "Compare these two images: what animals are they? How do the environments differ? Which would make a better wallpaper, and why?",
)
print(f"[visual tokens: {n_vis}]\n{reply}")

## Exercises

1. Screenshot your phone's settings page and ask the model to locate the "Wi-Fi" entry's bbox — feel the GUI-grounding precision.
2. Photograph a handwritten note and compare this model's transcription against a chapter-02 specialist (DeepSeek-OCR class).
3. Rerun everything with `Qwen3-VL-2B` and note the quality gaps — which tasks does the small model already handle?
4. Video understanding (needs more VRAM): swap the content entry for `{"type": "video", "video": "path.mp4"}` and try video QA.

**Next stop**: [02_api_frontier.ipynb](../../24-applications/notebooks/02_api_frontier.ipynb) — the same task set against three closed frontier APIs.